# GeoPilot TCC — processamento altimétrico no Google Colab

Este notebook reproduz o processamento automatizado utilizado no TCC, sem exigir a instalação local do ambiente Python. Ele utiliza o script oficial do repositório e gera o perfil altimétrico, os pontos amostrados e o resumo da rota.

**Entradas:** uma rota em GeoPackage (`.gpkg`) e um Modelo Digital de Elevação (`.tif`). Para arquivos grandes, recomenda-se armazená-los no Google Drive.

## 1. Obter o código e instalar as dependências

In [1]:
!git clone -q https://github.com/willigfreitas/geopilot-tcc-altimetria.git
%cd geopilot-tcc-altimetria
!python -m pip install -q -r requirements.txt
print("Ambiente preparado com sucesso.")

/content/geopilot-tcc-altimetria
Ambiente preparado com sucesso.


## 2. Selecionar a forma de acesso aos dados

Escolha **uma** das opções abaixo. Para um MDE grande, use o Google Drive.

### Opção A — enviar os arquivos pelo navegador

In [2]:
from google.colab import files

print("Selecione a rota em formato GPKG.")
rota_enviada = files.upload()
ROTA = next(iter(rota_enviada))

print("Selecione o MDE em formato TIF.")
mde_enviado = files.upload()
MDE = next(iter(mde_enviado))

print(f"Rota: {ROTA}")
print(f"MDE: {MDE}")

Selecione a rota em formato GPKG.


Saving rota_centro_aquarius_tcc_20261023.gpkg to rota_centro_aquarius_tcc_20261023.gpkg
Selecione o MDE em formato TIF.


Saving mde_sjc_recortado_31983_ii.tif to mde_sjc_recortado_31983_ii.tif
Rota: rota_centro_aquarius_tcc_20261023.gpkg
MDE: mde_sjc_recortado_31983_ii.tif


### Opção B — utilizar arquivos armazenados no Google Drive

Execute esta célula no lugar da Opção A e ajuste os dois caminhos.

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')
# ROTA = '/content/drive/MyDrive/GeoPilot/dados/rota.gpkg'
# MDE = '/content/drive/MyDrive/GeoPilot/dados/mde.tif'
# print(f"Rota: {ROTA}")
# print(f"MDE: {MDE}")

## 3. Definir os parâmetros e executar

In [4]:
from pathlib import Path
import subprocess
import sys

SRC_CALCULO = 'EPSG:31983'
ESPACAMENTO_M = 30.0
SAIDA = Path('/content/geopilot_resultado')

comando = [
    sys.executable,
    'tools/geopilot_geo_route.py',
    '--routes', str(ROTA),
    '--dem', str(MDE),
    '--output', str(SAIDA),
    '--crs', SRC_CALCULO,
    '--sample-spacing', str(ESPACAMENTO_M),
]
subprocess.run(comando, check=True)

CompletedProcess(args=['/usr/bin/python3', 'tools/geopilot_geo_route.py', '--routes', 'rota_centro_aquarius_tcc_20261023.gpkg', '--dem', 'mde_sjc_recortado_31983_ii.tif', '--output', '/content/geopilot_resultado', '--crs', 'EPSG:31983', '--sample-spacing', '30.0'], returncode=0)

## 4. Conferir os resultados

In [5]:
import json
import pandas as pd
from IPython.display import display

resumo = json.loads((SAIDA / 'resumo_rota.json').read_text(encoding='utf-8'))
perfil = pd.read_csv(SAIDA / 'perfil_altimetrico.csv')

display(pd.DataFrame({
    'Indicador': [
        'Distância (km)', 'Amostras', 'Cota inicial (m)',
        'Cota final (m)', 'Subida acumulada (m)',
        'Descida acumulada (m)', 'Declividade absoluta média (%)'
    ],
    'Valor': [
        resumo['distance_km'], resumo['sample_count'],
        resumo['initial_elevation_m'], resumo['final_elevation_m'],
        resumo['accumulated_ascent_m'], resumo['accumulated_descent_m'],
        resumo['mean_absolute_grade_percent']
    ]
}))
display(perfil.head())

,Indicador,Valor
0,Distância (km),4.031443
1,Amostras,136.000000
2,Cota inicial (m),605.500000
3,Cota final (m),604.000000
4,Subida acumulada (m),98.666667
5,Descida acumulada (m),100.166667
6,Declividade absoluta média (%),4.922812


,sample_id,distance_m,elevation_m,elevation_smooth_m,segment_distance_m,elevation_change_m,grade_percent,positive_ascent_m,accumulated_ascent_m,negative_descent_m,accumulated_descent_m,x,y
0,1,0.0,607.0,605.500000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,409228.177576,7.435625e+06
1,2,30.0,604.0,605.000000,30.0,-0.500000,-1.666667,0.0,0.0,0.500000,0.500000,409199.577388,7.435617e+06
2,3,60.0,604.0,604.000000,30.0,-1.000000,-3.333333,0.0,0.0,1.000000,1.500000,409174.992534,7.435601e+06
3,4,90.0,604.0,604.000000,30.0,0.000000,0.000000,0.0,0.0,0.000000,1.500000,409145.331254,7.435597e+06
4,5,120.0,604.0,603.333333,30.0,-0.666667,-2.222222,0.0,0.0,0.666667,2.166667,409115.669974,7.435592e+06


## 5. Baixar as saídas

In [6]:
import shutil
from google.colab import files

arquivo_zip = shutil.make_archive('/content/geopilot_resultado', 'zip', SAIDA)
files.download(arquivo_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Observações de reprodutibilidade

- O ambiente temporário do Colab é apagado ao encerrar a sessão; baixe as saídas ou copie-as para o Google Drive.
- O resultado depende do mesmo arquivo de rota, do mesmo MDE, do SRC de cálculo e do espaçamento de amostragem.
- O notebook não substitui o script: ele funciona como uma interface para executar a versão testada e versionada no GitHub.